##連接到Google Drive，會需要登入帳戶

In [7]:
# 連結到google雲端硬碟
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 處理train_info和train_data，轉成可以訓練模型的特徵。
資料處理整個跑完約需20分鐘

In [8]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from scipy.stats import kurtosis, skew
from scipy.signal import welch, find_peaks
from scipy.fft import rfft

# 設定檔案路徑
INFO_CSV_PATH   = "/content/drive/MyDrive/AI cup 程式繳交/train_info.csv" # train_info.csv的資料路徑
TRAIN_DATA_DIR  = "/content/drive/MyDrive/AI cup 程式繳交/train_data" # train_data資料夾的路徑
OUTPUT_CSV_PATH = "/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv" #存放處理完的train特徵路徑，建議設在「AI cup 程式繳交」資料夾裡

FS = 85  # 取樣頻率 85 Hz

def parse_cut_points(s):
    try:
        return np.array([int(x) for x in s.strip("[]\n").split()], dtype=int)
    except:
        return np.array([], dtype=int)

def load_sensor_file(fp):
    return np.loadtxt(fp)

def cut_swings(data, cp):
    swings = []
    for i in range(len(cp)-1):
        s,e = cp[i], cp[i+1]
        if e> s and e<= len(data):
            swings.append(data[s:e])
    return swings

# 頻譜質心 (spectral centroid)
def spectral_centroid(sig):
    f, Pxx = welch(sig, fs=FS)
    num = (f * Pxx).sum()
    den = Pxx.sum() + 1e-12
    return num / den

def spectral_entropy(sig):
    f,Pxx = welch(sig, fs=FS)
    Pxx /= Pxx.sum()
    return -np.sum(Pxx * np.log2(Pxx+1e-12))

def dominant_freq(sig):
    return np.argmax(np.abs(rfft(sig)))

def extract_features(swings):
    # 如果沒有任何 Swing，回傳全零
    if not swings:
        return np.zeros(73, dtype=float)

    ax_max, ay_ax, az_ax = [],[],[]
    gx_ax, gy_ax, gz_ax = [],[],[]
    ax_std, ay_std, az_std = [],[],[]
    gx_std, gy_std, gz_std = [],[],[]
    ax_m7, ay_m7, az_m7 = [],[],[]
    gx_m7, gy_m7, gz_m7 = [],[],[]
    kur, skw, ent = [],[],[]
    ax_rng, ay_rng, az_rng = [],[],[]
    gx_rng, gy_rng, gz_rng = [],[],[]
    gx_m, gy_m, gz_m = [],[],[]
    ener, slen = [],[]
    a_mag_m, g_std_m = [],[]
    ax_cv = []
    g_dom = []
    gyro_mag_m = []
    axgx_prod = []
    ax_peak, gy_std2, gyro_mag_peak = [],[],[]
    ax_skew, gx_kurt_v, az_ent = [],[],[]
    energy_diff = []
    gx_mean_diff = []
    gmag_mean_diff = []
    jerk_ax_v, jerk_ay_v, jerk_az_v = [],[],[]
    jerk_mag_v = []
    band_lo, band_md, band_hi = [],[],[]
    ax_peak_cnt = []
    p25_mag, p50_mag, p75_mag = [],[],[]
    corr_axgx, corr_aygy, corr_azgz = [],[],[]
    corr_axay, corr_axaz, corr_ayaz = [],[],[]
    spec_centroid = []
    second_peak = []

    for i, sw in enumerate(swings):
        if len(sw) < 7: continue
        ax,ay,az = sw[:,0], sw[:,1], sw[:,2]
        gx,gy,gz = sw[:,3], sw[:,4], sw[:,5]
        idx = np.argmax(ax)
        win = lambda x: x[max(0,idx-3):min(len(x),idx+4)]

        ax_max.append(ax[idx]); ay_ax.append(ay[idx]); az_ax.append(az[idx])
        gx_ax.append(gx[idx]); gy_ax.append(gy[idx]); gz_ax.append(gz[idx])
        ax_std.append(ax.std()); ay_std.append(ay.std()); az_std.append(az.std())
        gx_std.append(gx.std()); gy_std.append(gy.std()); gz_std.append(gz.std())
        ax_m7.append(win(ax).mean()); ay_m7.append(win(ay).mean()); az_m7.append(win(az).mean())
        gx_m7.append(win(gx).mean()); gy_m7.append(win(gy).mean()); gz_m7.append(win(gz).mean())
        ax_rng.append(ax.max()-ax.min()); ay_rng.append(ay.max()-ay.min()); az_rng.append(az.max()-az.min())
        gx_rng.append(gx.max()-gx.min()); gy_rng.append(gy.max()-gy.min()); gz_rng.append(gz.max()-gz.min())
        gx_m.append(gx.mean()); gy_m.append(gy.mean()); gz_m.append(gz.mean())
        all_s = np.hstack([ax,ay,az,gx,gy,gz])
        kur.append(kurtosis(all_s)); skw.append(skew(all_s)); ent.append(spectral_entropy(all_s))
        ener.append((all_s**2).sum()); slen.append(len(sw))
        mag_a = np.sqrt(ax**2+ay**2+az**2); a_mag_m.append(mag_a.mean())
        g_std_m.append(np.mean([gx.std(), gy.std(), gz.std()]))
        ax_cv.append(ax.std()/(abs(ax.mean())+1e-6))
        g_dom.append(dominant_freq(gx)); gyro_mag_m.append(np.sqrt(gx**2+gy**2+gz**2).mean())
        axgx_prod.append(ax.std()*gx.std())

        ax_peak.append(ax.max()); gy_std2.append(gy.std())
        gyro_mag_peak.append(np.sqrt(gx**2+gy**2+gz**2).max())
        ax_skew.append(skew(ax)); gx_kurt_v.append(kurtosis(gx)); az_ent.append(spectral_entropy(az))
        energy_diff.append((all_s**2).sum() - (ax**2).sum())
        if i>0:
            prev = swings[i-1];
            gx_mean_diff.append(gx.mean() - prev[:,3].mean())
            prev_mag = np.sqrt(prev[:,3]**2+prev[:,4]**2+prev[:,5]**2)
            gmag_mean_diff.append(mag_a.mean()-prev_mag.mean())

        jax = np.diff(ax)*FS; jay=np.diff(ay)*FS; jaz=np.diff(az)*FS
        jerk_ax_v.append(jax.mean()); jerk_ay_v.append(jay.mean()); jerk_az_v.append(jaz.mean())
        jerk_mag_v.append(np.diff(mag_a, prepend=mag_a[0]).mean()*FS)
        f,Pxx = welch(mag_a, fs=FS)
        band_lo.append(Pxx[f<=5].sum()); band_md.append(Pxx[(f>5)&(f<=15)].sum()); band_hi.append(Pxx[f>15].sum())
        ax_peak_cnt.append(len(find_peaks(ax)[0]))

        p25,p50,p75 = np.percentile(mag_a, [25,50,75])
        p25_mag.append(p25); p50_mag.append(p50); p75_mag.append(p75)
        corr_axgx.append(np.corrcoef(ax,gx)[0,1])
        corr_aygy.append(np.corrcoef(ay,gy)[0,1])
        corr_azgz.append(np.corrcoef(az,gz)[0,1])
        corr_axay.append(np.corrcoef(ax,ay)[0,1])
        corr_axaz.append(np.corrcoef(ax,az)[0,1])
        corr_ayaz.append(np.corrcoef(ay,az)[0,1])
        spec_centroid.append(spectral_centroid(mag_a))
        fftmag = np.abs(rfft(mag_a))
        if len(fftmag)>1:
            second_peak.append(np.sort(fftmag)[-2])
        else:
            second_peak.append(0.0)

    # 聚合成最後 feature 向量
    feats = []
    feats += [
        np.mean(ax_max),np.mean(ay_ax),np.mean(az_ax),
        np.mean(ax_std),np.mean(ay_std),np.mean(az_std),
        np.mean(ax_m7),np.mean(ay_m7),np.mean(az_m7),
        np.mean(gx_ax),np.mean(gy_ax),np.mean(gz_ax),
        np.mean(gx_std),np.mean(gy_std),np.mean(gz_std),
        np.mean(gx_m7),np.mean(gy_m7),np.mean(gz_m7),
        np.mean(kur),np.mean(skw),np.mean(ent),
        np.mean(ax_rng),np.mean(ay_rng),np.mean(az_rng),
        np.mean(gx_rng),np.mean(gy_rng),np.mean(gz_rng),
        np.mean(gx_m),np.mean(gy_m),np.mean(gz_m),
        np.mean(ener),np.mean(slen),
        np.mean(a_mag_m),np.mean(g_std_m),
        np.mean(ax_cv),
        np.mean(g_dom),np.mean(gyro_mag_m),np.mean(axgx_prod)
    ]
    feats += [
        np.mean(ax_peak),np.std(gy_std2),np.mean(gyro_mag_peak),
        np.mean(ax_skew),np.mean(gx_kurt_v),np.mean(az_ent),
        np.mean(energy_diff),
        np.mean(gx_mean_diff) if gx_mean_diff else 0.0,
        np.mean(gmag_mean_diff) if gmag_mean_diff else 0.0
    ]
    feats += [
        np.mean(jerk_ax_v),np.std(jerk_ax_v),
        np.mean(jerk_ay_v),np.std(jerk_ay_v),
        np.mean(jerk_az_v),np.std(jerk_az_v),
        np.mean(jerk_mag_v),np.std(jerk_mag_v),
        np.mean(band_lo),np.mean(band_md),np.mean(band_hi),
        np.mean(ax_peak_cnt),np.std(ax_peak_cnt)
    ]
    feats += [
        np.mean(p25_mag),np.mean(p50_mag),np.mean(p75_mag),
        np.mean(corr_axgx),np.mean(corr_aygy),np.mean(corr_azgz),
        np.mean(corr_axay),np.mean(corr_axaz),np.mean(corr_ayaz),
        np.mean(spec_centroid),np.mean(second_peak),
        np.mean(np.array(slen)/FS), np.std(np.array(slen)/FS)
    ]
    assert len(feats)==73
    return np.array(feats, dtype=float)

def get_sample_feature(fp, cp):
    if not os.path.exists(fp):
        return np.zeros(73, dtype=float)
    try:
        data = load_sensor_file(fp)
        swings = cut_swings(data, cp)
        return extract_features(swings)
    except:
        return np.zeros(73, dtype=float)

def main():
    df = pd.read_csv(INFO_CSV_PATH)
    df.columns = ['unique_id','player_id','mode','gender','hold_hand','play_years','level','cut_point']
    df['cut_point'] = df['cut_point'].apply(parse_cut_points)

    all_feats = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        filepath = os.path.join(TRAIN_DATA_DIR, f"{row.unique_id}.txt")
        all_feats.append(get_sample_feature(filepath, row.cut_point))

    feat_cols_basic = [
        'Ax_max','Ay_at_Axmax','Az_at_Axmax',
        'Ax_std','Ay_std','Az_std',
        'Ax_mean7','Ay_mean7','Az_mean7',
        'Gx_max_at_Axmax','Gy_max_at_Axmax','Gz_max_at_Axmax',
        'Gx_std','Gy_std','Gz_std',
        'Gx_mean7','Gy_mean7','Gz_mean7',
        'kurtosis','skewness','spectral_entropy',
        'Ax_range','Ay_range','Az_range',
        'Gx_range','Gy_range','Gz_range',
        'Gx_mean','Gy_mean','Gz_mean',
        'signal_energy','avg_swing_length',
        'A_magnitude_mean','G_std_mean',
        'Ax_cv','G_dominant_freq','gyro_magnitude_mean','Ax_std_times_Gx_std',
        'ax_peak','gy_std_std','gyro_mag_peak',
        'ax_skew','gx_kurt','az_entropy',
        'energy_diff','gx_mean_diff','gyro_mag_diff',
        'jerk_ax_mean','jerk_ax_std',
        'jerk_ay_mean','jerk_ay_std',
        'jerk_az_mean','jerk_az_std',
        'jerk_mag_mean','jerk_mag_std',
        'band_power_0_5Hz','band_power_5_15Hz','band_power_above_15Hz',
        'ax_peak_count_mean','ax_peak_count_std'
    ]
    feat_cols_extra = [
        'p25_mag','p50_mag','p75_mag',
        'corr_axgx','corr_aygy','corr_azgz',
        'corr_axay','corr_axaz','corr_ayaz',
        'spectral_centroid','second_peak',
        'swing_dur_mean','swing_dur_std'
    ]
    feat_cols = feat_cols_basic + feat_cols_extra
    assert len(feat_cols)==73

    features_df = pd.DataFrame(np.vstack(all_feats), columns=feat_cols)
    out = pd.concat([df.drop(columns=['cut_point']), features_df], axis=1)
    out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f" 輸出 {out.shape[0]} 筆，{len(feat_cols)} 個特徵 -> {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()

  0%|          | 0/1955 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 61, using nperseg = 61
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 62, using nperseg = 62
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
  0%|          | 1/1955 [00:00<24:42,  1.32it/s]/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 74, using nperseg = 74
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 75, using nperseg = 75
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


 輸出 1955 筆，73 個特徵 -> /content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv


## 處理test_info和test_data，轉成可以用來預測的特徵。
##函式和程式基本上和train_info和train_data一樣。唯一不同的是讀檔的位置是不一樣的。

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from scipy.stats import kurtosis, skew
from scipy.signal import welch, find_peaks
from scipy.fft import rfft


# 設定檔案路徑
INFO_CSV_PATH = "/content/drive/MyDrive/AI cup 程式繳交/test_info.csv" # test_info.csv的資料路徑
TRAIN_DATA_DIR = "/content/drive/MyDrive/AI cup 程式繳交/test_data" # test_data的資料夾路徑
OUTPUT_CSV_PATH = "/content/drive/MyDrive/AI cup 程式繳交/test_with_features_73.csv" # 存放處理完的test特徵路徑，建議設在「AI cup 程式繳交」資料夾裡

FS = 85  # 取樣頻率 85 Hz

def parse_cut_points(s):
    try:
        return np.array([int(x) for x in s.strip("[]\n").split()], dtype=int)
    except:
        return np.array([], dtype=int)

def load_sensor_file(fp):
    return np.loadtxt(fp)

def cut_swings(data, cp):
    swings = []
    for i in range(len(cp)-1):
        s,e = cp[i], cp[i+1]
        if e> s and e<= len(data):
            swings.append(data[s:e])
    return swings

def spectral_centroid(sig):
    f, Pxx = welch(sig, fs=FS)
    num = (f * Pxx).sum()
    den = Pxx.sum() + 1e-12
    return num / den

def spectral_entropy(sig):
    f,Pxx = welch(sig, fs=FS)
    Pxx /= Pxx.sum()
    return -np.sum(Pxx * np.log2(Pxx+1e-12))


def dominant_freq(sig):
    return np.argmax(np.abs(rfft(sig)))


def extract_features(swings):
    # 如果沒有任何 Swing，回傳全零
    if not swings:
        return np.zeros(73, dtype=float)

    ax_max, ay_ax, az_ax = [],[],[]
    gx_ax, gy_ax, gz_ax = [],[],[]
    ax_std, ay_std, az_std = [],[],[]
    gx_std, gy_std, gz_std = [],[],[]
    ax_m7, ay_m7, az_m7 = [],[],[]
    gx_m7, gy_m7, gz_m7 = [],[],[]
    kur, skw, ent = [],[],[]
    ax_rng, ay_rng, az_rng = [],[],[]
    gx_rng, gy_rng, gz_rng = [],[],[]
    gx_m, gy_m, gz_m = [],[],[]
    ener, slen = [],[]
    a_mag_m, g_std_m = [],[]
    ax_cv = []
    g_dom = []
    gyro_mag_m = []
    axgx_prod = []
    ax_peak, gy_std2, gyro_mag_peak = [],[],[]
    ax_skew, gx_kurt_v, az_ent = [],[],[]
    energy_diff = []
    gx_mean_diff = []
    gmag_mean_diff = []
    jerk_ax_v, jerk_ay_v, jerk_az_v = [],[],[]
    jerk_mag_v = []
    band_lo, band_md, band_hi = [],[],[]
    ax_peak_cnt = []
    p25_mag, p50_mag, p75_mag = [],[],[]
    corr_axgx, corr_aygy, corr_azgz = [],[],[]
    corr_axay, corr_axaz, corr_ayaz = [],[],[]
    spec_centroid = []
    second_peak = []

    for i, sw in enumerate(swings):
        if len(sw) < 7: continue
        ax,ay,az = sw[:,0], sw[:,1], sw[:,2]
        gx,gy,gz = sw[:,3], sw[:,4], sw[:,5]
        idx = np.argmax(ax)
        win = lambda x: x[max(0,idx-3):min(len(x),idx+4)]
        ax_max.append(ax[idx]); ay_ax.append(ay[idx]); az_ax.append(az[idx])
        gx_ax.append(gx[idx]); gy_ax.append(gy[idx]); gz_ax.append(gz[idx])
        ax_std.append(ax.std()); ay_std.append(ay.std()); az_std.append(az.std())
        gx_std.append(gx.std()); gy_std.append(gy.std()); gz_std.append(gz.std())
        ax_m7.append(win(ax).mean()); ay_m7.append(win(ay).mean()); az_m7.append(win(az).mean())
        gx_m7.append(win(gx).mean()); gy_m7.append(win(gy).mean()); gz_m7.append(win(gz).mean())
        ax_rng.append(ax.max()-ax.min()); ay_rng.append(ay.max()-ay.min()); az_rng.append(az.max()-az.min())
        gx_rng.append(gx.max()-gx.min()); gy_rng.append(gy.max()-gy.min()); gz_rng.append(gz.max()-gz.min())
        gx_m.append(gx.mean()); gy_m.append(gy.mean()); gz_m.append(gz.mean())
        all_s = np.hstack([ax,ay,az,gx,gy,gz])
        kur.append(kurtosis(all_s)); skw.append(skew(all_s)); ent.append(spectral_entropy(all_s))
        ener.append((all_s**2).sum()); slen.append(len(sw))
        mag_a = np.sqrt(ax**2+ay**2+az**2); a_mag_m.append(mag_a.mean())
        g_std_m.append(np.mean([gx.std(), gy.std(), gz.std()]))
        ax_cv.append(ax.std()/(abs(ax.mean())+1e-6))
        g_dom.append(dominant_freq(gx)); gyro_mag_m.append(np.sqrt(gx**2+gy**2+gz**2).mean())
        axgx_prod.append(ax.std()*gx.std())
        ax_peak.append(ax.max()); gy_std2.append(gy.std())
        gyro_mag_peak.append(np.sqrt(gx**2+gy**2+gz**2).max())
        ax_skew.append(skew(ax)); gx_kurt_v.append(kurtosis(gx)); az_ent.append(spectral_entropy(az))
        energy_diff.append((all_s**2).sum() - (ax**2).sum())
        if i>0:
            prev = swings[i-1];
            gx_mean_diff.append(gx.mean() - prev[:,3].mean())
            prev_mag = np.sqrt(prev[:,3]**2+prev[:,4]**2+prev[:,5]**2)
            gmag_mean_diff.append(mag_a.mean()-prev_mag.mean())
        jax = np.diff(ax)*FS; jay=np.diff(ay)*FS; jaz=np.diff(az)*FS
        jerk_ax_v.append(jax.mean()); jerk_ay_v.append(jay.mean()); jerk_az_v.append(jaz.mean())
        jerk_mag_v.append(np.diff(mag_a, prepend=mag_a[0]).mean()*FS)
        f,Pxx = welch(mag_a, fs=FS)
        band_lo.append(Pxx[f<=5].sum()); band_md.append(Pxx[(f>5)&(f<=15)].sum()); band_hi.append(Pxx[f>15].sum())
        ax_peak_cnt.append(len(find_peaks(ax)[0]))

        # 1. magnitude 百分位數
        p25,p50,p75 = np.percentile(mag_a, [25,50,75])
        p25_mag.append(p25); p50_mag.append(p50); p75_mag.append(p75)
        # 2. 軸間相關性
        corr_axgx.append(np.corrcoef(ax,gx)[0,1])
        corr_aygy.append(np.corrcoef(ay,gy)[0,1])
        corr_azgz.append(np.corrcoef(az,gz)[0,1])
        corr_axay.append(np.corrcoef(ax,ay)[0,1])
        corr_axaz.append(np.corrcoef(ax,az)[0,1])
        corr_ayaz.append(np.corrcoef(ay,az)[0,1])
        # 3. spectral centroid
        spec_centroid.append(spectral_centroid(mag_a))
        # 4. second-largest FFT peak magnitude
        fftmag = np.abs(rfft(mag_a))
        if len(fftmag)>1:
            second_peak.append(np.sort(fftmag)[-2])
        else:
            second_peak.append(0.0)

    # 聚合成最後 feature 向量
    feats = []
    feats += [
        np.mean(ax_max),np.mean(ay_ax),np.mean(az_ax),
        np.mean(ax_std),np.mean(ay_std),np.mean(az_std),
        np.mean(ax_m7),np.mean(ay_m7),np.mean(az_m7),
        np.mean(gx_ax),np.mean(gy_ax),np.mean(gz_ax),
        np.mean(gx_std),np.mean(gy_std),np.mean(gz_std),
        np.mean(gx_m7),np.mean(gy_m7),np.mean(gz_m7),
        np.mean(kur),np.mean(skw),np.mean(ent),
        np.mean(ax_rng),np.mean(ay_rng),np.mean(az_rng),
        np.mean(gx_rng),np.mean(gy_rng),np.mean(gz_rng),
        np.mean(gx_m),np.mean(gy_m),np.mean(gz_m),
        np.mean(ener),np.mean(slen),
        np.mean(a_mag_m),np.mean(g_std_m),
        np.mean(ax_cv),
        np.mean(g_dom),np.mean(gyro_mag_m),np.mean(axgx_prod)
    ]
    feats += [
        np.mean(ax_peak),np.std(gy_std2),np.mean(gyro_mag_peak),
        np.mean(ax_skew),np.mean(gx_kurt_v),np.mean(az_ent),
        np.mean(energy_diff),
        np.mean(gx_mean_diff) if gx_mean_diff else 0.0,
        np.mean(gmag_mean_diff) if gmag_mean_diff else 0.0
    ]
    feats += [
        np.mean(jerk_ax_v),np.std(jerk_ax_v),
        np.mean(jerk_ay_v),np.std(jerk_ay_v),
        np.mean(jerk_az_v),np.std(jerk_az_v),
        np.mean(jerk_mag_v),np.std(jerk_mag_v),
        np.mean(band_lo),np.mean(band_md),np.mean(band_hi),
        np.mean(ax_peak_cnt),np.std(ax_peak_cnt)
    ]
    feats += [
        np.mean(p25_mag),np.mean(p50_mag),np.mean(p75_mag),
        np.mean(corr_axgx),np.mean(corr_aygy),np.mean(corr_azgz),
        np.mean(corr_axay),np.mean(corr_axaz),np.mean(corr_ayaz),
        np.mean(spec_centroid),np.mean(second_peak),
        np.mean(np.array(slen)/FS), np.std(np.array(slen)/FS)
    ]
    assert len(feats)==73
    return np.array(feats, dtype=float)

def get_sample_feature(fp, cp):
    if not os.path.exists(fp):
        return np.zeros(73, dtype=float)
    try:
        data = load_sensor_file(fp)
        swings = cut_swings(data, cp)
        return extract_features(swings)
    except:
        return np.zeros(73, dtype=float)

def main():
    df = pd.read_csv(INFO_CSV_PATH)
    df.columns = ['unique_id', 'mode', 'cut_point']
    df['cut_point'] = df['cut_point'].apply(parse_cut_points)

    features = []
    print("開始處理每筆資料 ...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        uid = row['unique_id']
        txt_path = os.path.join(TRAIN_DATA_DIR, f"{uid}.txt")
        cut_points = row['cut_point']
        feat = get_sample_feature(txt_path, cut_points)
        features.append(feat)

    features_array = np.array(features)
    feat_cols_basic = [
        'Ax_max','Ay_at_Axmax','Az_at_Axmax',
        'Ax_std','Ay_std','Az_std',
        'Ax_mean7','Ay_mean7','Az_mean7',
        'Gx_max_at_Axmax','Gy_max_at_Axmax','Gz_max_at_Axmax',
        'Gx_std','Gy_std','Gz_std',
        'Gx_mean7','Gy_mean7','Gz_mean7',
        'kurtosis','skewness','spectral_entropy',
        'Ax_range','Ay_range','Az_range',
        'Gx_range','Gy_range','Gz_range',
        'Gx_mean','Gy_mean','Gz_mean',
        'signal_energy','avg_swing_length',
        'A_magnitude_mean','G_std_mean',
        'Ax_cv','G_dominant_freq','gyro_magnitude_mean','Ax_std_times_Gx_std',
        'ax_peak','gy_std_std','gyro_mag_peak',
        'ax_skew','gx_kurt','az_entropy',
        'energy_diff','gx_mean_diff','gyro_mag_diff',
        'jerk_ax_mean','jerk_ax_std',
        'jerk_ay_mean','jerk_ay_std',
        'jerk_az_mean','jerk_az_std',
        'jerk_mag_mean','jerk_mag_std',
        'band_power_0_5Hz','band_power_5_15Hz','band_power_above_15Hz',
        'ax_peak_count_mean','ax_peak_count_std'
    ]
    feat_cols_extra = [
        'p25_mag','p50_mag','p75_mag',
        'corr_axgx','corr_aygy','corr_azgz',
        'corr_axay','corr_axaz','corr_ayaz',
        'spectral_centroid','second_peak',
        'swing_dur_mean','swing_dur_std'
    ]
    # 合併名稱
    feat_cols = feat_cols_basic + feat_cols_extra
    assert len(feat_cols)==73

    features_df = pd.DataFrame(features_array, columns=feat_cols)
    df_final = pd.concat([df.drop(columns=['cut_point']), features_df], axis=1)
    df_final.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Done! 特徵檔案已輸出到：{OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()


開始處理每筆資料 ...


  0%|          | 0/1430 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 95, using nperseg = 95
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 96, using nperseg = 96
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
  0%|          | 1/1430 [00:00<05:03,  4.72it/s]/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 99, using nperseg = 99
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,
  0%|          | 2/1430 [00:00<04:41,  5.08it/s]/usr/local/lib/python3.11/dist-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 80, using nperseg = 80
  freqs, _, Pxy = _spect

##進入訓練之前，要先確認是否安裝好xgboost

In [ ]:
!pip install xgboost


## 測試性別，將訓練資料80/20分，80％為訓練，20％為驗證


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑

# 選擇特徵和目標變數
X = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand','play_years', 'level'])  # 預測 gender
y = df['gender'] - 1  # gender 從 1,2 轉為 0,1

# 分割資料集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立並訓練模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)
model.fit(X_train_scaled, y_train)

# 預測與評估
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ 預測準確率：{accuracy:.4f}\n")
print("📊 分類報告：")
print(classification_report(y_test, y_pred, target_names=["male", "female"]))

# 混淆矩陣
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=["male", "female"], yticklabels=["male", "female"])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# 特徵重要性
importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(importance_df)


## 預測性別的真實資料（使用全部的X_train資料）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑
x_test = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/test_with_features_73.csv') # test_with_features_73.csv的檔案路徑
x_test = x_test.drop(columns=['unique_id'])
# 選擇特徵和目標變數
X_train= df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand','play_years', 'level'])  # 預測 gender
y_train = df['gender'] - 1  # gender 從 1,2 轉為 0,1

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(x_test)

# 建立並訓練模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)
model.fit(X_train_scaled, y_train)

# 預測
y_pred_gender = model.predict_proba(X_test_scaled)[:, 0]
y_pred_gender

## 手測試，將訓練資料80/20分，80％為訓練，20％為驗證

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑

# 選擇特徵和目標變數
X = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand','play_years', 'level'])  # 拿掉 label 欄位
y = df['hold_hand'] -1  # 預測 hold_hand（0：左手，1：右手）

# 分割資料集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立並訓練模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)
model.fit(X_train_scaled, y_train)

# 預測與評估
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"✅ hold_hand 預測準確率：{acc:.4f}")
print("📊 分類報告：")
print(classification_report(y_test, y_pred, target_names=["right-handed", "left-handed"]))
# 特徵重要性
importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=["right-handed", "left-handed"], yticklabels=["right-handed", "left-handed"])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# 繪製圖表
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'][:10], importance_df['Importance'][:10])
plt.xlabel('Importance')
plt.title('Feature Importance for hold_hand')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 輸出重要性
print(importance_df)

## 預測手的真實資料（使用全部的X_train資料）


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑
x_test = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/test_with_features_73.csv') # test_with_features_73.csv的檔案路徑
X_test = x_test.drop(columns=['unique_id'])
# 選擇特徵和目標變數
X_train = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand','play_years', 'level'])  # 拿掉 label 欄位
y_train = df['hold_hand'] -1  # 預測 hold_hand（0：左手，1：右手）

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立並訓練模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)
model.fit(X_train_scaled, y_train)

# 預測
y_pred_hold_hand = model.predict_proba(X_test_scaled)[:, 0]
y_pred_hold_hand

## 使用train_with_features測試 play_years，將訓練資料80/20分，80％為訓練，20％為驗證


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑

# 準備資料
X = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand','play_years', 'level'])
y = df['play_years']

# 分割資料集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)

# 訓練模型
model.fit(X_train_scaled, y_train)

# 預測
y_pred = model.predict(X_test_scaled)

# 計算準確率
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ play_years 預測準確率：{accuracy:.4f}")
print("📊 分類報告：")
print(classification_report(y_test, y_pred, target_names=["play_years: 0", "play_years: 1","play_years: 2"]))
# 混淆矩陣
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix for play_years")
plt.show()

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'][:10], importance_df['Importance'][:10])
plt.xlabel('Importance')
plt.title('Feature Importance for play_years')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
# 特徵重要性
importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

print(importance_df)



## 預測play_years的真實資料（使用全部的X_train資料）

In [ ]:
# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑
x_test = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/test_with_features_73.csv') # test_with_features_73.csv的檔案路徑
X_test = x_test.drop(columns=['unique_id'])
# 準備資料
X_train = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand','play_years', 'level'])
y_train = df['play_years']

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)

# 訓練模型
model.fit(X_train_scaled, y_train)

# 預測
y_pred_play_years = model.predict_proba(X_test_scaled)
y_pred_play_years

## 使用train_with_features測試 level，將訓練資料80/20分，80％為訓練，20％為驗證


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑

# 預測目標：level（2~5）
X = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand', 'play_years', 'level'])
y = df['level'] - 2  # 轉換成 0, 1, 2, 3

# 分割資料集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立並訓練模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)
model.fit(X_train_scaled, y_train)

# 預測
y_pred = model.predict(X_test_scaled)

# 預測準確率
accuracy = (y_pred == y_test).mean()

# 混淆矩陣
labels = [2, 3, 4, 5]  # 原始 level
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix for Level Prediction")
plt.show()

importances = model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'][:10], importance_df['Importance'][:10])
plt.xlabel('Importance')
plt.title('Feature Importance for play_years')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
print(importance_df)

print(f"✅ level 預測準確率：{accuracy:.4f}")
print("\n📋 各等級 precision / recall / f1-score：")
print(classification_report(y_test, y_pred, target_names=[f'Level {i}' for i in labels]))


## 預測level的真實資料（使用全部的X_train資料）

In [ ]:
# 讀取資料
df = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/train_with_features_73.csv') # train_with_features_73.csv的檔案路徑
x_test = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/test_with_features_73.csv') # test_with_features_73.csv的檔案路徑
X_test = x_test.drop(columns=['unique_id'])
# 預測目標：level（2~5）
X_train = df.drop(columns=['unique_id', 'player_id', 'gender', 'hold_hand', 'play_years', 'level'])
y_train = df['level'] - 2  # 轉換成 0, 1, 2, 3

# 特徵標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立並訓練模型
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    random_state=42,
    eval_metric="mlogloss",
    use_label_encoder=False
)
model.fit(X_train_scaled, y_train)

# 預測
y_pred_level = model.predict_proba(X_test_scaled)
y_pred_level

##將所有的預測結果打包，輸出成csv檔案。

In [ ]:
x_test = pd.read_csv('/content/drive/MyDrive/AI cup 程式繳交/test_with_features_73.csv') # train_with_features_73.csv的檔案路徑
play_years_first_column = y_pred_play_years[:, 0]
play_years_second_column = y_pred_play_years[:, 1]
play_years_third_column = y_pred_play_years[:, 2]
level_first_column = y_pred_level[:, 0]
level_second_column = y_pred_level[:, 1]
level_third_column = y_pred_level[:, 2]
level_fourth_column = y_pred_level[:, 3]
final_prediction = pd.DataFrame({'unique_id': x_test['unique_id'], 'gender': y_pred_gender, 'hold racket handed':y_pred_hold_hand, 'play years_0':play_years_first_column, 'play years_1':play_years_second_column, 'play years_2': play_years_third_column,'level_2':level_first_column ,'level_3':level_second_column ,'level_4':level_third_column ,'level_5': level_fourth_column})
final_prediction.to_csv('/content/drive/MyDrive/AI cup 程式繳交/final_prediction.csv', index=False) #存放最後的預測結果路徑，建議設在「AI cup 程式繳交」資料夾裡
